## A/B Testing Analysis for E-Commerce


### Mount Google Drive dan Setup Path

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')

## Load Dataset
PROJECT_PATH = '/content/drive/MyDrive/Portfolio/ab-testing-analysis'
RAW_PATH = f'{PROJECT_PATH}/data/raw'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
IMAGE_PATH = f'{PROJECT_PATH}/images'

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(IMAGE_PATH, exist_ok=True)

print('Project path:', PROJECT_PATH)

ValueError: mount failed

### Data Load dan Quality Check

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

sns.set_theme(style='whitegrid')

file_path = f'{RAW_PATH}/ab_data.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
print(df.head())
print(df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

### Group Balance Check

In [ ]:
# Cek konsistensi group vs landing_page
mismatch = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
]
print('Baris tidak konsisten:', len(mismatch))

# Buang baris tidak konsisten
df = df.drop(mismatch.index)
print('Shape setelah buang mismatch:', df.shape)

In [ ]:
# Cek dan buang duplikat user_id (satu user = satu observasi)
print('Duplikat user_id:', df['user_id'].duplicated().sum())
df = df.drop_duplicates(subset='user_id', keep='first')
print('Shape setelah dedup:', df.shape)

In [ ]:
# Pastikan dua kelompok seimbang
group_counts = df['group'].value_counts()
print('Group distribution:')
print(group_counts)
print(f'\nRatio: {group_counts.iloc[0]/group_counts.iloc[1]:.2f}')

# Convert group ke label yang lebih jelas
df['group_label'] = df['group'].map({0: 'control', 1: 'treatment'}) if df['group'].dtype == 'int64' else df['group']
print(df['group_label'].value_counts())

### Conversion Rate Comparison

In [ ]:
# Conversion rate per group
cr = df.groupby('group_label')['converted'].agg(['mean', 'count', 'sum'])
cr.columns = ['conversion_rate', 'total_users', 'total_conversions']
print(cr)

# Funnel analysis
for grp in df['group_label'].unique():
    subset = df[df['group_label'] == grp]
    total = len(subset)
    converted = subset['converted'].sum()
    rate = converted / total
    print(f'{grp}: {total} users → {converted} conversions → CR: {rate:.4f}')

### Visualization Funnel

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar chart: conversion rate by group
cr_plot = cr.reset_index()
sns.barplot(data=cr_plot, x='group_label', y='conversion_rate', ax=axes[0],
            palette=['#3498db', '#e74c3c'], hue='group_label', legend=False)
axes[0].set_title('Conversion Rate by Group')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_ylim(0, cr_plot['conversion_rate'].max() * 1.3)

# Add value labels on bars
for i, row in cr_plot.iterrows():
    axes[0].text(i, row['conversion_rate'] + 0.002, f"{row['conversion_rate']:.4f}",
                 ha='center', fontsize=11, fontweight='bold')

# Pie charts for each group
for idx, grp in enumerate(['control', 'treatment']):
    subset = df[df['group_label'] == grp]
    conv = subset['converted'].sum()
    not_conv = len(subset) - conv
    axes[idx + 1].pie([conv, not_conv],
                      labels=['Converted', 'Not Converted'],
                      autopct='%1.1f%%',
                      colors=['#2ecc71', '#e74c3c'],
                      startangle=90)
    axes[idx + 1].set_title(f'{grp.capitalize()} Group')

plt.tight_layout()
plt.savefig(f'{IMAGE_PATH}/ab_testing_funnel.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Validasi akhir setelah cleaning
assert df['user_id'].is_unique
assert df['group_label'].isin(['control', 'treatment']).all()

print('Jumlah user unik:', df['user_id'].nunique())
print('Duplikat user_id:', df['user_id'].duplicated().sum())

### Statistical Hypothesis Testing

In [ ]:
# Define groups
control = df[df['group_label'] == 'control']
treatment = df[df['group_label'] == 'treatment']

conversions = [control['converted'].sum(), treatment['converted'].sum()]
nobs = [len(control), len(treatment)]

# 1. Proportion Z-Test
z_stat, p_value = proportions_ztest(conversions, nobs, alternative='two-sided')
print('=== Proportion Z-Test ===')
print(f'Z-statistic: {z_stat:.4f}')
print(f'P-value: {p_value:.6f}')

# 2. Chi-Square Test
contingency = pd.crosstab(df['group_label'], df['converted'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'\n=== Chi-Square Test ===')
print(f'Chi2: {chi2:.4f}')
print(f'P-value: {p_chi:.6f}')
print(f'Degrees of freedom: {dof}')

# 3. Confidence Interval
cr_control = control['converted'].mean()
cr_treatment = treatment['converted'].mean()
diff = cr_treatment - cr_control

se_control = np.sqrt(cr_control * (1 - cr_control) / len(control))
se_treatment = np.sqrt(cr_treatment * (1 - cr_treatment) / len(treatment))
se_diff = np.sqrt(se_control**2 + se_treatment**2)

ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f'\n=== Confidence Interval ===')
print(f'Control CR:    {cr_control:.4f}')
print(f'Treatment CR:  {cr_treatment:.4f}')
print(f'Difference:    {diff:.4f}')
print(f'95% CI:        [{ci_lower:.4f}, {ci_upper:.4f}]')

### Practical Significance & Summary

In [ ]:
alpha = 0.05
relative_lift = (diff / cr_control) * 100

# Konversi nilai ke percentage points untuk kejelasan
diff_pp = diff * 100
ci_lower_pp = ci_lower * 100
ci_upper_pp = ci_upper * 100

print('=== Practical Significance ===')
print(f'Control conversion rate:   {cr_control:.4%}')
print(f'Treatment conversion rate: {cr_treatment:.4%}')
print(f'Absolute difference:       {diff_pp:.4f} percentage points')
print(f'Relative lift:             {relative_lift:.2f}%')
print(f'P-value:                   {p_value:.6f}')
print(f'95% CI:                    [{ci_lower_pp:.4f}, {ci_upper_pp:.4f}] percentage points')

if p_value < alpha and diff > 0:
    conclusion = (
        'Treatment menunjukkan peningkatan yang signifikan secara statistik. '
        'Pertimbangkan rollout setelah evaluasi biaya dan risiko.'
    )
elif p_value < alpha and diff < 0:
    conclusion = (
        'Treatment menunjukkan penurunan yang signifikan secara statistik. '
        'Jangan melakukan rollout.'
    )
else:
    conclusion = (
        'Tidak terdapat perbedaan conversion rate yang signifikan. '
        'Jangan melakukan rollout berdasarkan hasil eksperimen ini; '
        'pertimbangkan eksperimen lanjutan jika diperlukan.'
    )

print(f'\nCalculation: {cr_treatment:.4%} - {cr_control:.4%} = {diff_pp:.4f} percentage points')
print('\nRecommendation:', conclusion)

In [ ]:
print(
    pd.crosstab(
        df['group_label'],
        df['landing_page']
    )
)

### Parsing dan Validasi Timestamp

In [ ]:
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    errors='coerce'
)

print('Invalid timestamp:', df['timestamp'].isna().sum())
print('Experiment period:', df['timestamp'].min(), 'to', df['timestamp'].max())

### SQL Query

In [ ]:
# Instalasi cukup dilakukan satu kali di awal notebook
!pip -q install duckdb

import duckdb
import pandas as pd

# Pastikan timestamp bertipe datetime
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    errors='coerce'
)

# Validasi data final sebelum masuk ke DuckDB
assert df['user_id'].is_unique
assert df['converted'].isin([0, 1]).all()
assert set(df['group_label'].dropna().unique()) == {
    'control',
    'treatment'
}
assert df['timestamp'].notna().all()

print('Jumlah user unik:', df['user_id'].nunique())
print('Duplikat user_id:', df['user_id'].duplicated().sum())
print('Invalid timestamp:', df['timestamp'].isna().sum())
print(
    'Periode eksperimen:',
    df['timestamp'].min(),
    'sampai',
    df['timestamp'].max()
)

# Validasi konsistensi group dan landing page
print('\n=== Group vs Landing Page ===')
print(pd.crosstab(df['group_label'], df['landing_page']))

# Daftar dataframe final ke DuckDB
con = duckdb.connect()
con.register('ab_testing', df)

# Conversion summary
sql_summary = con.sql('''
    SELECT
        group_label,
        COUNT(*) AS total_users,
        SUM(converted) AS total_conversions,
        ROUND(AVG(converted) * 100.0, 4) AS conversion_rate_pct
    FROM ab_testing
    GROUP BY group_label
    ORDER BY group_label
''').df()

print('\n=== Conversion Summary ===')
print(sql_summary)

# Jumlah user per hari
daily_counts = con.sql('''
    SELECT
        CAST(timestamp AS DATE) AS test_date,
        COUNT(*) AS users
    FROM ab_testing
    GROUP BY CAST(timestamp AS DATE)
    ORDER BY test_date
''').df()

print('\n=== Daily User Counts ===')
print(daily_counts)

# Daily conversion trend
sql_daily = con.sql('''
    SELECT
        CAST(timestamp AS DATE) AS test_date,
        group_label,
        COUNT(*) AS daily_users,
        SUM(converted) AS daily_conversions,
        ROUND(AVG(converted) * 100.0, 4) AS daily_cr_pct
    FROM ab_testing
    GROUP BY CAST(timestamp AS DATE), group_label
    ORDER BY test_date, group_label
''').df()

print('\n=== Daily Conversion Trend ===')
print(sql_daily)

In [ ]:
# ==========================================
# Save A/B Testing Results to Google Drive
# ==========================================

# Pastikan folder output tersedia
os.makedirs(PROCESSED_PATH, exist_ok=True)

# 1. Simpan dataset bersih untuk Tableau
columns = [
    'user_id',
    'timestamp',
    'group_label',
    'landing_page',
    'converted'
]

ab_testing_clean = df[columns].copy()

ab_testing_clean = ab_testing_clean.rename(
    columns={'group_label': 'group'}
)

ab_testing_clean.to_csv(
    f'{PROCESSED_PATH}/ab_testing_clean.tsv',
    sep='\t',
    index=False,
    encoding='utf-8'
)

print(ab_testing_clean.columns.tolist())

print(open(
    f'{PROCESSED_PATH}/ab_testing_clean.tsv',
    encoding='utf-8'
).readline())

# 2. Simpan conversion summary
conversion_summary = (
    df.groupby('group_label')
      .agg(
          total_users=('user_id', 'nunique'),
          total_conversions=('converted', 'sum'),
          conversion_rate=('converted', 'mean')
      )
      .reset_index()
)

conversion_summary['conversion_rate_pct'] = (
    conversion_summary['conversion_rate'] * 100
)

conversion_summary.to_csv(
    f'{PROCESSED_PATH}/conversion_summary.csv',
    index=False
)

# 3. Simpan tren conversion harian
daily_conversion = (
    df.assign(test_date=df['timestamp'].dt.date)
      .groupby(['test_date', 'group_label'])
      .agg(
          daily_users=('user_id', 'nunique'),
          daily_conversions=('converted', 'sum'),
          daily_conversion_rate=('converted', 'mean')
      )
      .reset_index()
)

daily_conversion['daily_conversion_rate_pct'] = (
    daily_conversion['daily_conversion_rate'] * 100
)

daily_conversion.to_csv(
    f'{PROCESSED_PATH}/daily_conversion_trend.csv',
    index=False
)

# 4. Simpan hasil statistical testing
statistical_results = pd.DataFrame({
    'metric': [
        'control_conversion_rate',
        'treatment_conversion_rate',
        'absolute_difference',
        'absolute_difference_percentage_points',
        'relative_lift_percent',
        'z_statistic',
        'z_test_p_value',
        'chi_square_statistic',
        'chi_square_p_value',
        'confidence_interval_lower_percentage_points',
        'confidence_interval_upper_percentage_points',
        'alpha'
    ],
    'value': [
        cr_control,
        cr_treatment,
        diff,
        diff_pp,
        relative_lift,
        z_stat,
        p_value,
        chi2,
        p_chi,
        ci_lower_pp,
        ci_upper_pp,
        alpha
    ]
})

statistical_results.to_csv(
    f'{PROCESSED_PATH}/statistical_results.csv',
    index=False
)

# 5. Simpan contingency table
group_landing_check = pd.crosstab(
    df['group_label'],
    df['landing_page']
)

group_landing_check.to_csv(
    f'{PROCESSED_PATH}/group_landing_page_check.csv'
)

# 6. Simpan daftar file output
print('=== Files Saved ===')
print(f'{PROCESSED_PATH}/ab_testing_clean.tsv')
print(f'{PROCESSED_PATH}/conversion_summary.csv')
print(f'{PROCESSED_PATH}/daily_conversion_trend.csv')
print(f'{PROCESSED_PATH}/statistical_results.csv')
print(f'{PROCESSED_PATH}/group_landing_page_check.csv')
print(f'{IMAGE_PATH}/ab_testing_funnel.png')
print('Kolom:', ab_testing_clean.columns.tolist())
print('Shape:', ab_testing_clean.shape)
print(
    pd.read_csv(
        f'{PROCESSED_PATH}/ab_testing_clean.tsv',
        sep='\t'
    ).head()
)

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

!zip -r /content/ab_testing_export.zip \
  /content/drive/MyDrive/Portfolio/ab-testing-analysis

In [ ]:
!ls -lh /content/ab_testing_export.zip

In [ ]:
files.download('/content/ab_testing_export.zip')